# Predominance Diagrams and Electrochemical Surface Coverage


## Goal of the notebook

Let us use potential and pH to find the most stable candidate state for a catalyst. Then we will connect adsorption free energy with equilibrium coverage. FePc and CoPc join our Pt(111) example.


## Description of the method

A predominance diagram shows the candidate with the lowest formation grand free energy at each potential and pH. A finite temperature model gives probabilities for the empty catalyst and its adsorbed states. We will use one shared state table for Pt(111), FePc, and CoPc. The Pt(111) extension also introduces site degeneracy and lateral interactions.


## What we will do

1. Let us explore a Pt(111) potential and pH map.
2. We will compare FePc and CoPc maps and coverages on SHE and RHE scales.
3. We will connect adsorption free energy with Langmuir coverage.
4. We will explore Pt(111) site competition and interactions as extensions.


In [ ]:
# Install dependencies only in Colab; otherwise use the course environment.
!if python -c "import google.colab" >/dev/null 2>&1; then pip install -q numpy pandas matplotlib ipywidgets; fi

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from ipywidgets import Dropdown, FloatSlider, HBox, interactive_output, VBox
from matplotlib.colors import ListedColormap

In [ ]:
# Standard publication figure sizes in inches
plt.rc("figure", figsize  = (8.25 / 2.54, 8.25 / 2.54))
plt.rc("figure", dpi      = 100)
plt.rc("lines",  linewidth= 4)
plt.rc("font",   size     = 10)
plt.rc("font",   family   = "sans-serif")
plt.rc("axes",   titlesize= 10)
plt.rc("axes",   labelsize= 9)
plt.rc("xtick",  labelsize= 8)
plt.rc("ytick",  labelsize= 8)
plt.rc("legend", fontsize = 8)
plt.rc("figure", titlesize= 10)

pd.options.display.float_format = "{:.2f}".format
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]


In [ ]:
boltzmann_constant_ev_k = 8.617333262145e-5  # Boltzmann constant in eV/K


## Predominance diagrams: the basic idea

**Core exercise**

Let us ask one thermodynamic question: **which candidate surface state has the lowest free energy at each potential and pH?** This is like finding the lowest line among potential dependent free energies.

For state $i$, define its formation grand free energy relative to the clean surface:

$$\Omega_i(U_{\mathrm{RHE}})=\Delta G_i^0+
u_iU_{\mathrm{RHE}}.$$


Here, $\Delta G_i^0$ is the formation free energy at 0 V versus RHE. The signed CHE coefficient $
u_i$ follows the formation reaction. Forming H* consumes one $\mathrm{H^+}+e^-$ pair, so $
u=+1$. Forming OH* or O* from water releases one or two pairs, so $
u=-1$ or $-2$.

The predominant state is

$$i_{\mathrm{pred}}(U,\mathrm{pH})=\operatorname*{arg\,min}_i\Omega_i.$$

States $i$ and $j$ meet where $\Omega_i=\Omega_j$.


$$U_{\mathrm{RHE}}=U_{\mathrm{SHE}}+\frac{k_BT\ln 10}{e}\,\mathrm{pH}.$$

At 298 K, the shift is about 0.059 V per pH unit. A transition at fixed RHE potential therefore appears as a sloped boundary on a SHE potential and pH map.

The map displays the lowest candidate only. It gives no mixed coverage, metastability, activation barrier, or rate. Its result depends on the states in the table. For Pt(111), each species uses its listed lowest energy adsorption geometry.


In [ ]:
pt111_predominance_states = pd.DataFrame([
    ('empty', 0.0, 0),
    ('H*', -0.27, 1),
    ('H2O*', 0.27, 0),
    ('OH*', 1.02, -1),
    ('O*', 1.23, -2),
], columns=['state', 'dG0_ev', 'nu'])
pt111_predominance_states

In [ ]:
def predominance_map(table, U_SHE_values, pH_values):
    nernst_slope_v = boltzmann_constant_ev_k * 298.0 * np.log(10.0)
    U_SHE_grid, pH_grid = np.meshgrid(U_SHE_values, pH_values)
    U_RHE_grid = U_SHE_grid + nernst_slope_v * pH_grid
    grand_energies_ev = np.stack([ row.dG0_ev + row.nu * U_RHE_grid for row in table.itertuples(index=False) ])
    return np.argmin(grand_energies_ev, axis=0), U_SHE_grid, pH_grid

In [ ]:
U_SHE_values = np.linspace(-0.40, 1.20, 401)
pH_values = np.linspace(0.0, 14.0, 281)
predominant, U_SHE_grid, pH_grid = predominance_map(pt111_predominance_states, U_SHE_values, pH_values)

In [ ]:
def plot_predominance_point(potential_she_v, pH):
    fig, ax = plt.subplots()
    ax.pcolormesh(U_SHE_grid, pH_grid, predominant, shading="auto", cmap=ListedColormap(colors), vmin=-0.5, vmax=4.5)
    handles = [plt.Line2D([], [], marker="s", linestyle="", markersize=8, color=colors[i], label=state) for i, state in enumerate(pt111_predominance_states.state)]
    ax.plot(potential_she_v, pH, marker="x", color="black", markersize=9)
    ax.set_xlabel("U / V vs SHE")
    ax.set_ylabel("pH")
    ax.set_title("Pt(111) thermodynamic predominance")
    ax.legend(handles=handles, loc="upper right")
    fig.tight_layout()
    plt.show()
    slope = boltzmann_constant_ev_k * 298.0 * np.log(10)
    potential_rhe_v = potential_she_v + slope * pH
    energies = pt111_predominance_states.dG0_ev + pt111_predominance_states.nu * potential_rhe_v
    # .iloc selects by position; argmin gives the position of the lowest energy.
    state = pt111_predominance_states.state.iloc[int(np.argmin(energies))]
    print(f"Selected point: {potential_rhe_v:.2f} V vs RHE; predominant state: {state}")

In [ ]:
predominance_controls = {"potential_she_v": FloatSlider(value=0.80, min=-0.40, max=1.20, step=0.01, description="U / V vs SHE", continuous_update=False), "pH": FloatSlider(value=0.0, min=0, max=14, step=0.1, description="pH", continuous_update=False),}
predominance_output = interactive_output(plot_predominance_point, predominance_controls)
display(HBox([predominance_output, VBox(list(predominance_controls.values()))]))

In [ ]:
present_indices = np.unique(predominant)
print("States present:", ", ".join(pt111_predominance_states.state.iloc[present_indices]))
print("An absent candidate is never the lowest energy state in this plotted window.")

## Build a general state table

**Core exercise**

Let us describe each candidate state relative to the empty catalyst and water. Count protons and electrons released in its formation:

$$\Omega_i(E_{\mathrm{SHE}},\mathrm{pH})=\Delta G_i^0-n_{e,i}eE_{\mathrm{SHE}}-n_{\mathrm{H},i}k_BT\ln(10)\,\mathrm{pH}.$$

Positive counts represent release; negative counts represent consumption. Energies are in eV per modeled site, so $eE$ has the same numerical value as potential in V. Water activity is one, the empty state has zero formation energy, and temperature is 298 K.


### Check the signs before the map

| State | Formation relative to the empty catalyst | $n_e$ | $n_{\mathrm{H}}$ |
|:---|:---|---:|---:|
| H* | $*+\mathrm{H}^+ + e^- \rightarrow \mathrm{H}*$ | −1 | −1 |
| OH* | $*+\mathrm{H_2O} \rightarrow \mathrm{OH}* + \mathrm{H}^+ + e^-$ | 1 | 1 |
| O* | $*+\mathrm{H_2O} \rightarrow \mathrm{O}* + 2\mathrm{H}^+ + 2e^-$ | 2 | 2 |
| OOH* | $*+2\mathrm{H_2O} \rightarrow \mathrm{OOH}* + 3\mathrm{H}^+ + 3e^-$ | 3 | 3 |

For these states, $n_e=n_{\mathrm{H}}$, so $\Omega_i=\Delta G_i^0-n_{e,i}eE_{\mathrm{RHE}}$. This matches the CHE form with $
u_i=-n_{e,i}$. A pure electron transfer has $n_{\mathrm{H}}=0$; a pure proton transfer has $n_e=0$. The general equation covers each case.


### Read a boundary

Two states meet when their grand free energies are equal. For $n_{e,i}
e n_{e,j}$, solve for the SHE potential:

$$E_{ij,\mathrm{SHE}}=\frac{\Delta G_i^0-\Delta G_j^0-(n_{\mathrm{H},i}-n_{\mathrm{H},j})k_BT\ln(10)\,\mathrm{pH}}{(n_{e,i}-n_{e,j})e}.$$

The map shows this boundary where the two states sit below the other candidates. A third state may cover their crossing. When electron counts match, solve the equality for pH.


In [ ]:
temperature_k = 298.0
thermal_energy_ev = boltzmann_constant_ev_k * temperature_k
nernst_slope_v = thermal_energy_ev * np.log(10.0)

# One editable table per catalyst; keep every energy on the same reference.
# Molecular values are the associative inputs already used in notebook 2.3.
molecular_states = {
    "FePc": [("empty", 0.0, 0, 0), ("OH*", 0.78, 1, 1),
             ("O*", 1.90, 2, 2), ("OOH*", 3.85, 3, 3)],
    "CoPc-CNT": [("empty", 0.0, 0, 0), ("OH*", 0.82, 1, 1),
                 ("O*", 1.63, 2, 2), ("OOH*", 4.00, 3, 3)],
}
state_columns = ["state", "dG0_ev", "n_e_released", "n_H_released"]
catalyst_states = {name: pd.DataFrame(rows, columns=state_columns)
                   for name, rows in molecular_states.items()}
# Convert the existing Pt convention without changing its input energies.
catalyst_states["Pt(111)"] = pt111_predominance_states.assign(
    n_e_released=-pt111_predominance_states.nu,
    n_H_released=-pt111_predominance_states.nu)[state_columns]
display(pd.concat(catalyst_states, names=["catalyst", "row"]).round(2))

### Read the molecular examples

The FePc and CoPc-CNT values form a teaching model, not an experimental stability dataset. FePc values are supplied for this course. CoPc-CNT (unmodified) values are read from the [source free-energy figure](https://www.nature.com/articles/s41929-023-00924-5?fromPaywallRec=false#Sec14). Each molecule has one metal site and four mutually exclusive states: empty, OH*, O*, and OOH*. The Pt(111) table also includes H* and H₂O*.

These maps describe the candidate adsorbates in the table. The model covers spin state, axial ligand, protonation, dissolution, and catalyst stability only when those states have energy entries. Regions outside water stability can appear because the map contains no water decomposition states. Read the figure as a conditional adsorbate map, not a complete Pourbaix diagram or ORR activity prediction.

To add a catalyst, enter the same four columns in `catalyst_states`: state, formation energy, and signed electron and proton counts. Include the empty state and use one reference for every energy. Add each state to `state_colors`, then rerun the control cell to update the menus. Convert energies to a shared reaction reference before you combine them.

The FePc and CoPc teaching inputs also appear in the [associative free energy examples](2.3_Free_energy_diagram_literature.ipynb).


In [ ]:
def state_free_energies(table, potential_v, pH, potential_scale="SHE"):
    # Broadcasting evaluates one point, a curve, or a complete map at once.
    potential, acidity = np.broadcast_arrays(np.asarray(potential_v, dtype=float), np.asarray(pH, dtype=float))
    if potential_scale == "RHE":
        potential = potential - nernst_slope_v * acidity
    elif potential_scale != "SHE":
        raise ValueError("Choose SHE or RHE.")
    shape = (-1) + (1) * potential.ndim
    # .to_numpy() converts a table column into a numerical array.
    g0 = table.dG0_ev.to_numpy().reshape(shape)
    electrons = table.n_e_released.to_numpy().reshape(shape)
    protons = table.n_H_released.to_numpy().reshape(shape)
    return g0 - electrons * potential - protons * nernst_slope_v * acidity

def state_probabilities(energies):
    # Subtract the minimum to avoid overflowing the exponential.
    weights = np.exp(-(energies - energies.min(axis=0)) / thermal_energy_ev)
    return weights / weights.sum(axis=0)

## Compare FePc and CoPc

Choose each catalyst, potential, pH, and reference scale. The marker shows the selected condition. The curves below show state probabilities at that pH. Each color marks the same state across catalysts.

**Predict first:** At 1.10 V vs SHE and pH 0, which adsorbate dominates each catalyst? At fixed SHE potential, how may a change in pH affect the state? What happens at fixed RHE potential?


In [ ]:
# Change plot ranges here. The controls use the same potential limits.
comparison_potential_limits = (-0.40, 1.80)
comparison_pH_limits = (0.0, 14.0)
comparison_potentials = np.linspace(*comparison_potential_limits, 441)
comparison_pH = np.linspace(*comparison_pH_limits, 281)
map_potential, map_pH = np.meshgrid(comparison_potentials, comparison_pH)
state_colors = dict(zip(pt111_predominance_states.state, colors))
state_colors["OOH*"] = "#AA4499"

In [ ]:
def plot_catalyst_maps(left_catalyst, right_catalyst, potential_scale, potential_v, pH):
    fig, axes = plt.subplots(1, 2, figsize=(17.0 / 2.54, 8.25 / 2.54), sharex=True, sharey=True)
    for ax, name in zip(axes, (left_catalyst, right_catalyst)):
        table = catalyst_states[name]
        energies = state_free_energies(table, map_potential, map_pH, potential_scale)
        # axis=0 compares candidate states at every potential and pH.
        regions = np.argmin(energies, axis=0)
        cmap = ListedColormap([state_colors[state] for state in table.state])
        ax.pcolormesh(map_potential, map_pH, regions, cmap=cmap, vmin=-0.5, vmax=len(table)-0.5, shading="auto", rasterized=True)
        ax.plot(potential_v, pH, marker="x", color="black", markersize=8)
        ax.set(xlabel=f"E / V vs {potential_scale}", title=name, xlim=comparison_potential_limits, ylim=comparison_pH_limits)
        ax.set_box_aspect(1)
        present = np.unique(regions)
        handles = [plt.Line2D([], [], marker="s", linestyle="", markersize=6, color=state_colors[table.state.iloc[i]], markeredgecolor="black", markeredgewidth=0.5, label=table.state.iloc[i]) for i in present]
        ax.legend(handles=handles, fontsize=7, loc="upper right", facecolor="white", edgecolor="none", framealpha=0.9)
    axes[0].set_ylabel("pH")
    fig.tight_layout()
    plt.show()

### From the minimum to a mixture

With equal state degeneracies and no interactions, each site fraction is

$$\theta_i=\frac{\exp[-\Omega_i/(k_BT)]}{\sum_j\exp[-\Omega_j/(k_BT)]},\qquad\sum_i\theta_i=1.$$

The sum includes the empty state. Near a boundary, several states can have appreciable probability even when the map displays one predominant state. The table reports each energy relative to the empty catalyst, each probability, and the state with the minimum energy.


In [ ]:
def plot_catalyst_coverages(left_catalyst, right_catalyst, potential_scale, potential_v, pH):
    fig, axes = plt.subplots(1, 2, figsize=(17.0 / 2.54, 8.25 / 2.54), sharex=True, sharey=True)
    records = []
    for ax, name in zip(axes, (left_catalyst, right_catalyst)):
        table = catalyst_states[name]
        energies = state_free_energies(table, comparison_potentials, pH, potential_scale)
        probabilities = state_probabilities(energies)
        for index, state in enumerate(table.state):
            ax.plot(comparison_potentials, probabilities[index], color=state_colors[state], label=state)
        ax.axvline(potential_v, color="black", linestyle="--", linewidth=1)
        ax.set(xlabel=f"E / V vs {potential_scale}", title=f"{name}, pH {pH:.2f}", xlim=comparison_potential_limits, ylim=(0, 1.02))
        ax.set_box_aspect(1)
        ax.legend(fontsize=7, frameon=False)
        selected = state_free_energies(table, potential_v, pH, potential_scale)
        result = table[["state"]].copy()
        result["Ω / eV"] = selected
        result["site fraction"] = state_probabilities(selected)
        result["predominant"] = np.arange(len(table)) == np.argmin(selected)
        result.insert(0, "catalyst", name)
        records.append(result)
    axes[0].set_ylabel("Equilibrium site fraction")
    fig.tight_layout()
    plt.show()
    display(pd.concat(records, ignore_index=True).round(2))

In [ ]:
comparison_controls = {"left_catalyst": Dropdown(options=list(catalyst_states), value="FePc", description="Left"), "right_catalyst": Dropdown(options=list(catalyst_states), value="CoPc-CNT", description="Right"), "potential_scale": Dropdown(options=["SHE", "RHE"], value="SHE", description="Reference"), "potential_v": FloatSlider(value=1.10, min=comparison_potential_limits[0], max=comparison_potential_limits[1], step=0.01, description="E / V", continuous_update=False), "pH": FloatSlider(value=0.0, min=comparison_pH_limits[0], max=comparison_pH_limits[1], step=0.1, description="pH", continuous_update=False),}
comparison_map_output = interactive_output(plot_catalyst_maps, comparison_controls)
comparison_coverage_output = interactive_output(plot_catalyst_coverages, comparison_controls)
display(VBox([VBox(list(comparison_controls.values())), comparison_map_output, comparison_coverage_output]))

### Read the comparison

At 1.10 V vs SHE and pH 0, these inputs favor OH* on FePc and O* on CoPc-CNT. The FePc OH* region spans about 0.78 to 1.12 V vs RHE. Its coverage curve shows how the site fraction changes across this interval.

On the SHE scale, pH shifts these boundaries. On the RHE scale, the boundaries stay fixed for the equal proton and electron counts in these examples. At the same numerical potential, SHE and RHE represent different conditions whenever pH is above zero. For one physical condition, use $E_{\mathrm{RHE}}=E_{\mathrm{SHE}}+0.059\times\mathrm{pH}$ at 298 K.


## Connect adsorption free energy with coverage

**Core exercise**

Coverage is the fraction of surface sites in a given state:

$$\theta_i=\frac{N_i}{N_{\mathrm{sites}}},\qquad\sum_i\theta_i=1.$$

Let us begin with one adsorbate, A, and an empty site:

$$\mathrm{A} + * \rightleftharpoons \mathrm{A}*.$$

If its potential dependent formation free energy is $\Delta G_A(U)$, the partition function is


$$Z=1+\exp[-\beta\Delta G_A(U)],\qquad\theta_A=\frac{\exp[-\beta\Delta G_A(U)]}{Z},\qquad\theta_*=\frac{1}{Z}.$$

This is the Langmuir isotherm in free energy form. The independent site model assumes equilibrium, one adsorbate per site, equivalent sites, and zero lateral interaction. Adsorbate activity is included in $\Delta G_A$:

$$\frac{\theta_A}{1-\theta_A}=K_Aa_A,\qquad K_A=\exp(-\beta\Delta G_A^0).$$

At $\Delta G_A=0$, equally degenerate empty and occupied states each have probability one half. As formation free energy falls, coverage rises.


In [ ]:
def langmuir_coverage(delta_G_ev):
    # Coverage of one adsorbate competing with one empty state.
    beta_ev = 1.0 / (boltzmann_constant_ev_k * 298.0)
    return 1.0 / (1.0 + np.exp(beta_ev * np.asarray(delta_G_ev)))

In [ ]:
def plot_langmuir_point(delta_G_ev):
    delta_G_grid_ev = np.linspace(-0.30, 0.30, 400)
    fig, ax = plt.subplots()
    ax.plot(delta_G_grid_ev, langmuir_coverage(delta_G_grid_ev), label="298 K")
    selected = float(langmuir_coverage(delta_G_ev))
    ax.scatter(delta_G_ev, selected, color=colors[0], zorder=3)
    ax.axvline(0, color="0.5", linestyle=":", linewidth=1)
    ax.set_xlabel(r"$\Delta G_A$ / eV")
    ax.set_ylabel(r"$\theta_A$")
    ax.set_ylim(0, 1)
    ax.legend()
    fig.tight_layout()
    plt.show()
    print(f"Selected coverage: {selected:.2f}")

In [ ]:
langmuir_controls = {"delta_G_ev": FloatSlider(value=0.0, min=-0.30, max=0.30, step=0.005, description="ΔG / eV", continuous_update=False)}
langmuir_output = interactive_output(plot_langmuir_point, langmuir_controls)
display(HBox([langmuir_output, VBox(list(langmuir_controls.values()))]))

> **At 298 K:** With equal degeneracy, the empty and occupied states have equal probability at $\Delta G_A=0$.

## Let potential shift CHE formation energies

Relative to an empty site, the formation energies in this notebook are:


The signs follow the formation reactions. H* consumes one $\mathrm{H^+}+e^-$ pair. OH* and O* form from water and release one or two pairs. On the RHE scale, the pH term is already part of the reference. For a fixed SHE potential, use

$$U_{\mathrm{RHE}}=U_{\mathrm{SHE}}+\frac{k_BT\ln10}{e}\,\mathrm{pH}.$$

If $\Delta G_A(U)=\Delta G_A^0+
u U$, the coverage ratio follows

$$\frac{\theta_A}{1-\theta_A}=\exp[-\beta(\Delta G_A^0+
u U)].$$

For equal degeneracy, half coverage occurs at


$$U_{1/2}=-\frac{\Delta G_A^0}{
u}.$$

Near room temperature, a 59 mV shift changes a one electron Boltzmann ratio by a factor of ten. A two electron coefficient doubles this potential sensitivity. This is why the ideal independent site curves change over a narrow potential range.


## Let Pt(111) sites compete

Pt(111) exposes top, bridge, fcc, and hcp geometries. These sites share surface atoms, so our model treats them as competing configurations in one representative cell.

The adsorption free energies come from [Kolb et al. (2014)](https://doi.org/10.1063/1.4869749), listed in the order **ontop, hcp, fcc, bridge**. The source lists O at top, hcp, and fcc sites. H₂O is neutral on the CHE scale, so its electron coefficient is zero.


In [ ]:
pt111_configurations = pd.DataFrame([
    ('H', 'top', 1, -0.24, 1),
    ('H', 'hcp', 1, -0.25, 1),
    ('H', 'fcc', 1, -0.27, 1),
    ('H', 'bridge', 3, -0.23, 1),
    ('H2O', 'top', 1, 0.28, 0),
    ('H2O', 'hcp', 1, 0.28, 0),
    ('H2O', 'fcc', 1, 0.27, 0),
    ('H2O', 'bridge', 3, 0.28, 0),
    ('OH', 'top', 1, 1.02, -1),
    ('OH', 'hcp', 1, 1.54, -1),
    ('OH', 'fcc', 1, 1.26, -1),
    ('OH', 'bridge', 3, 1.02, -1),
    ('O', 'top', 1, 2.51, -2),
    ('O', 'hcp', 1, 1.69, -2),
    ('O', 'fcc', 1, 1.23, -2),
], columns=['species', 'site', 'degeneracy', 'dG0_ev', 'nu'])
pt111_configurations

For mutually exclusive configurations $j$, the partition function is

$$Z=1+\sum_jg_j\exp[-\beta\Omega_j(U)],$$

where the first term represents an empty cell and $g_j$ is the degeneracy. Each configuration has energy

$$\Omega_j(U)=\Delta G_j^0+
u_jU.$$

Its probability is

$$p_j=\frac{g_j\exp[-\beta\Omega_j]}{Z}.$$

Degeneracy contributes entropy $-k_BT\ln g_j$. Several symmetry equivalent configurations can carry a combined probability comparable to a single configuration with a slightly lower energy.


Add configuration probabilities over sites to obtain species coverage. Add them over species to obtain site occupancy. The empty probability and all configuration probabilities sum to one.

This model gives a probability per representative cell. To report monolayer coverage, define the cell and count how many surface atoms each configuration occupies.


In [ ]:
def configuration_probabilities(table, potential_v, interaction_ev=0.0, tolerance=1e-12, max_iterations=1000):
    # Return equilibrium probabilities for one representative cell.
    beta_ev = 1.0 / (boltzmann_constant_ev_k * 298.0)

    def probabilities_at_coverage(theta):
        grand_energy_ev = (table.dG0_ev.to_numpy() + table.nu.to_numpy() * potential_v + interaction_ev * theta)
        log_weights = (np.log(table.degeneracy.to_numpy()) - beta_ev * grand_energy_ev)
        shift = max(0.0, float(log_weights.max()))
        weights = np.exp(log_weights - shift)
        empty_weight = np.exp(-shift)
        denominator = empty_weight + weights.sum()
        return weights / denominator, empty_weight / denominator

    # Bisection brackets a physical self-consistent root on [0, 1].
    # For attractive W, multiple roots can occur; this selects the bracketed root.
    lower_coverage = 0.0
    upper_coverage = 1.0
    for _ in range(max_iterations):
        theta = 0.5 * (lower_coverage + upper_coverage)
        probabilities, empty_probability = probabilities_at_coverage(theta)
        residual = float(probabilities.sum()) - theta
        if abs(residual) < tolerance:
            break
        if residual > 0.0:
            lower_coverage = theta
        else:
            upper_coverage = theta
    else:
        raise RuntimeError("Coverage iteration did not converge")

    result = table.copy()
    result["grand_energy_ev"] = (result.dG0_ev + result.nu * potential_v + interaction_ev * theta)
    result["probability"] = probabilities
    return result, float(empty_probability)

In [ ]:
example, empty_probability = configuration_probabilities(pt111_configurations, potential_v=0.80)
print(f"Empty cell probability at 0.80 V vs RHE: {empty_probability:.2f}")
example.sort_values("probability", ascending=False)

In [ ]:
def coverage_curves(table, potential_grid_v, interaction_ev=0.0):
    species_records = []
    site_records = []
    for potential_v in potential_grid_v:
        probabilities, empty = configuration_probabilities(table, potential_v, interaction_ev)
        by_species = probabilities.groupby("species").probability.sum()
        by_site = probabilities.groupby("site").probability.sum()
        species_records.append({"U_RHE": potential_v, "empty": empty, **by_species})
        site_records.append({"U_RHE": potential_v, **by_site})
    return (pd.DataFrame(species_records).fillna(0.0), pd.DataFrame(site_records).fillna(0.0))

In [ ]:
potential_grid_v = np.linspace(-0.25, 1.25, 301)
species_coverage, site_coverage = coverage_curves(pt111_configurations, potential_grid_v)

In [ ]:
def plot_configuration_selection(potential_v):
    fig, axes = plt.subplots(1, 2, figsize=(17.0 / 2.54, 8.25 / 2.54), sharey=True)
    for i, name in enumerate(["empty", "H", "H2O", "OH", "O"]):
        axes[0].plot(species_coverage.U_RHE, species_coverage[name], color=colors[i], label=name)
    for i, name in enumerate(["top", "hcp", "fcc", "bridge"]):
        axes[1].plot(site_coverage.U_RHE, site_coverage[name], color=colors[i], label=name)
    for ax in axes:
        ax.axvline(potential_v, color="black", linestyle=":", linewidth=1)
        ax.set_xlabel("U / V vs RHE")
        ax.set_ylabel("Probability per surface cell")
        ax.set_ylim(0, 1)
        ax.legend()
    axes[0].set_title("Coverage by species")
    axes[1].set_title("Occupied configuration by site")
    fig.tight_layout()
    plt.show()
    selected, empty = configuration_probabilities(pt111_configurations, potential_v)
    print(f"Selected potential: {potential_v:.2f} V vs RHE; empty probability: {empty:.2f}")

In [ ]:
configuration_controls = {"potential_v": FloatSlider(value=0.80, min=-0.25, max=1.25, step=0.01, description="U / V vs RHE", continuous_update=False)}
configuration_output = interactive_output(plot_configuration_selection, configuration_controls)
display(HBox([configuration_output, VBox(list(configuration_controls.values()))]))

## Compare with an experimental Pt(111) figure

![Capacitance curves generated from cyclic voltammograms of Pt(111)](https://www.researchgate.net/publication/357925522/figure/fig1/AS:1113954285821953@1642598806340/Capacitance-curves-generated-from-cyclic-voltammograms-of-Pt111-in-01-M-HClO-4-pH_W640.jpg)

Reference image: [ResearchGate figure](https://www.researchgate.net/publication/357925522/figure/fig1/AS:1113954285821953@1642598806340/Capacitance-curves-generated-from-cyclic-voltammograms-of-Pt111-in-01-M-HClO-4-pH_W640.jpg).

Let us compare the experimental potential response with the equilibrium state model.


## Add lateral interactions

In the independent cell model, each configuration responds to its own adsorption free energy. A mean field repulsion adds $W\theta$ to every occupied configuration:

$$\Omega_j(U,\theta)=\Delta G_j^0+
u_jU+W\theta.$$

The total occupied fraction also sets the energy, so we solve the model self consistently. Positive $W$ penalizes a crowded surface and can spread or shift coverage transitions. Negative $W$ favors collective occupation and can sharpen them.


> **Pause and predict:** As $W$ rises, which changes first: the identity of the lowest energy adsorbate or the potential where the surface fills?
>
> **What to look for:** The model adds a collective occupation penalty. It describes no ordered adlayer structure.


In [ ]:
def plot_coverage_model(interaction):
    species_coverage, _ = coverage_curves(pt111_configurations, potential_grid_v, interaction_ev=interaction)
    plt.figure()
    for index, column in enumerate(["empty", "H", "H2O", "OH", "O"]):
        plt.plot(species_coverage.U_RHE, species_coverage[column], color=colors[index], label=column)
    plt.xlabel("U / V vs RHE")
    plt.ylabel("Probability per surface cell")
    plt.ylim(0, 1)
    plt.title(f"T = 298 K, W = {interaction:.2f} eV")
    plt.legend()
    plt.show()

In [ ]:
interaction_controls = {"interaction": FloatSlider(value=0.12, min=-0.30, max=0.30, step=0.01, description="W / eV", continuous_update=False)}
interaction_output = interactive_output(plot_coverage_model, interaction_controls)
display(HBox([interaction_output, VBox(list(interaction_controls.values()))]))

## Student exercise: Compare FePc and CoPc

**Core exercise**

1. Predict the dominant adsorbate on each catalyst at 1.10 V vs SHE and pH 0. Check the map and probability table.
2. Hold the SHE potential fixed and change pH. Record each state change. Repeat at fixed RHE potential.
3. Find the narrow OH* region for FePc. Use the coverage curves to explain its site fraction.
4. Set $\Delta G=0$ in the Langmuir plot and explain the resulting coverage.

For a seminar extension, inspect the Pt(111) configuration and interaction plots.


In [ ]:
# Your code here: record a potential, pH, and predominant state for each catalyst.
# state_free_energies(catalyst_states["FePc"], 1.10, 0.0, "SHE")
# np.argmin gives the position of the minimum; .iloc selects that table row.

### Your answer

Prediction: [write your prediction]

Selected states and potentials: [record your result]

Chemical explanation: [explain the role of the electrode reference]

Assumption or limitation: [state one limitation]


## Seminar exercises

1. Enter corrected site energies and identify which coverage transitions shift.
2. Set each degeneracy to one. Explain how the curves respond.
3. Compare $W=-0.30$, $W=0$, and $W=+0.30$ eV at 298 K.
4. Explain how overlapping Pt(111) configurations differ from independent adsorption site lattices.


## What have you learned

Let us gather the main ideas from this notebook. You can now:

- Build a state table from formation energies and proton and electron counts.
- Explain how the electrode reference shapes a potential and pH map.
- Compare FePc and CoPc-CNT maps and coverages while keeping thermodynamic state separate from catalytic activity.
- Distinguish the predominant state from a mixed equilibrium coverage.
- Explain how site degeneracy and lateral interactions affect the model.
